![JohnSnowLabs](https://nlp.johnsnowlabs.com/assets/images/logo.png)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/JohnSnowLabs/visual-nlp-workshop/blob/master/tutorials/Certification_Trainings_JSL/01.03.Handwritten_Text_Recognition.ipynb)

If you are using the `spark-ocr` library, please use this [01.03.Handwritten_Text_Recognition](https://github.com/JohnSnowLabs/visual-nlp-workshop/blob/master/tutorials/Certification_Trainings/01.03.Handwritten_Text_Recognition.ipynb) notebook.

## Blogposts and videos

- [Text Detection in Spark OCR](https://medium.com/spark-nlp/text-detection-in-spark-ocr-dcd8002bdc97)

- [Table Detection & Extraction in Spark OCR](https://medium.com/spark-nlp/table-detection-extraction-in-spark-ocr-50765c6cedc9)

- [Extract Tabular Data from PDF in Spark OCR](https://medium.com/spark-nlp/extract-tabular-data-from-pdf-in-spark-ocr-b02136bc0fcb)

- [Signature Detection in Spark OCR](https://medium.com/spark-nlp/signature-detection-in-spark-ocr-32f9e6f91e3c)

- [GPU image pre-processing in Spark OCR](https://medium.com/spark-nlp/gpu-image-pre-processing-in-spark-ocr-3-1-0-6fc27560a9bb)

- [How to Setup Spark OCR on UBUNTU - Video](https://www.youtube.com/watch?v=cmt4WIcL0nI)


**More examples here**

https://github.com/JohnSnowLabs/spark-ocr-workshop

For getting the trial license please go to:

https://www.johnsnowlabs.com/install/

Please choose GPU runtime

### Colab Setup

In [ ]:
# Install the johnsnowlabs library to access Spark-OCR and Spark-NLP for Healthcare, Finance, and Legal.
!pip install -q johnsnowlabs

In [ ]:
from google.colab import files
print('Please Upload your John Snow Labs License using the button below')
license_keys = files.upload()

In [ ]:
from johnsnowlabs import nlp, visual

# After uploading your license run this to install all licensed Python Wheels and pre-download Jars the Spark Session JVM
nlp.install(refresh_install=True, visual=True)

⚠️ Important: running the next cell will **automatically restart** the Colab runtime (this is expected after installing new jars/wheels). Once it restarts, just continue running the cells below.

In [ ]:
import os
os.kill(os.getpid(), 9)

In [1]:
import importlib.resources

from pyspark.ml import PipelineModel
from pyspark.sql import functions as F

In [2]:
from johnsnowlabs import nlp, visual
import pandas as pd

# Automatically load license data and start a session with all jars user has access to
spark = nlp.start(visual=True, hardware_target="gpu")

👌 Launched gpu optimized session with with: 🚀Spark-NLP==6.4.0, 💊Spark-Healthcare==6.4.0, 🕶Spark-OCR==6.4.0, running on ⚡ PySpark==3.4.0


## Read image

In [3]:
!wget -q https://raw.githubusercontent.com/JohnSnowLabs/spark-ocr-workshop/Fix_handwritten_notebook/jupyter/data/handwritten/handwritten_example.jpg

In [4]:
image_example_df = spark.read.format("binaryFile").load("handwritten_example.jpg")
image_df = visual.BinaryToImage().transform(image_example_df).cache()

visual.display_images(image_df)

Output hidden; open in https://colab.research.google.com to view.

In [5]:
binary_to_image = visual.BinaryToImage()
#binary_to_image.setImageType(ImageType.TYPE_3BYTE_BGR)

text_detector = visual.ImageTextDetectorV2 \
    .pretrained("image_text_detector_v2", "en", "clinical/ocr") \
    .setInputCol("image") \
    .setOutputCol("text_regions") \
    .setWithRefiner(True) \
    .setSizeThreshold(10) \
    .setScoreThreshold(0.2) \
    .setTextThreshold(0.2) \
    .setLinkThreshold(0.3) \
    .setWidth(500)

ocr = visual.ImageToTextV2.pretrained("ocr_base_handwritten_v2_opt", "en", "clinical/ocr") \
    .setRegionsColumn("text_regions")\
    .setInputCols(["image"]) \
    .setGroupImages(True) \
    .setOutputCol("text")

# .setRotated(True) removed: deprecated since spark-ocr 5.4.1 -- rotated
# vs. regular coordinates are now auto-detected, so no functionality is lost.
draw_regions = visual.ImageDrawRegions() \
    .setInputCol("image") \
    .setInputRegionsCol("text_regions") \
    .setOutputCol("image_with_regions") \
    .setRectColor(visual.Color.green)

pipeline = PipelineModel(stages=[
    binary_to_image,
    text_detector,
    ocr,
    draw_regions
])

image_text_detector_v2 download started this may take some time.
Approximate size to download 75.3 MB


## Run pipeline and show results

In [6]:
result = pipeline.transform(image_example_df).cache()
visual.display_images(result, "image_with_regions")
print(("").join([x.text for x in result.select("text").collect()]))

Output hidden; open in https://colab.research.google.com to view.